<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/DBMS_RAG_SQLite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with a Local DBMS: Text-to-SQL using SQLite

This notebook demonstrates **Retrieval-Augmented Generation (RAG)** where the knowledge source is a relational database rather than a document store.

**How it works:**
1. Build a SQLite database from the classic **Supplier-Parts** schema (DDL from a `.sql` file, data from an `.xlsx` file).
2. Extract the database **schema** as structured context.
3. Given a natural-language question, the LLM generates a **SQL query**.
4. Execute the SQL and feed the **results back** to the LLM for a natural-language answer.

**Learning goals:**
- Understand how RAG applies beyond vector databases
- See how schema metadata serves as retrieval context
- Practice grounding LLM output on real query results
- Compare answers with and without database context

**Provider setup:** Gemini is tried first; if quota is exhausted, Ollama Cloud is used automatically.  
Store `GEMINI_API_KEY` and/or `OLLAMA_API_KEY` in Colab Secrets (or a local `.env` file).

In [1]:
%pip install -q -U google-genai openai openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 20.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


## 1) Imports and Provider Helpers

SQLite is built into Python — no extra database driver needed.  
We use **openpyxl** to load data from the Excel file.

The provider helpers below are identical to the pattern used in the RAG countries notebook:
Gemini first, Ollama Cloud fallback, keys from Colab Secrets or `.env`.

In [2]:
import os
import sqlite3
import re
from pathlib import Path

import openpyxl
from google import genai
from openai import OpenAI

# --- Load .env for local VS Code testing ---
env_path = Path.cwd() / ".env"
if env_path.exists():
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

# --- Colab Secrets ---
try:
    from google.colab import userdata  # type: ignore
except Exception:
    userdata = None


def _colab_secret(name):
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None


def get_gemini_api_key():
    return os.getenv("GEMINI_API_KEY") or _colab_secret("GEMINI_API_KEY")


def get_ollama_api_key():
    return os.getenv("OLLAMA_API_KEY") or _colab_secret("OLLAMA_API_KEY")


def get_ollama_base_url():
    return os.getenv("OLLAMA_BASE_URL", "https://ollama.com/v1")


def get_ollama_model(default_model="deepseek-v3.1:671b-cloud"):
    return os.getenv("OLLAMA_MODEL", default_model)


def has_llm_provider():
    return bool(get_gemini_api_key() or get_ollama_api_key())


def is_gemini_quota_error(exc):
    msg = str(exc).lower()
    return any(s in msg for s in ("quota", "resource_exhausted", "429", "rate limit", "exceeded your current quota"))


def generate_text(prompt, system_prompt=None, gemini_model="gemini-2.5-flash", ollama_model=None):
    gemini_key = get_gemini_api_key()
    ollama_key = get_ollama_api_key()
    ollama_url = get_ollama_base_url()

    if gemini_key:
        try:
            client = genai.Client(api_key=gemini_key)
            contents = prompt if not system_prompt else f"{system_prompt}\n\n{prompt}"
            resp = client.models.generate_content(model=gemini_model, contents=contents)
            return resp.text, "gemini"
        except Exception as exc:
            if not is_gemini_quota_error(exc):
                raise
            if not ollama_key:
                raise RuntimeError(
                    "Gemini quota exhausted and OLLAMA_API_KEY is not set. "
                    "Add your Ollama Cloud key as a Colab Secret named OLLAMA_API_KEY."
                ) from exc
            print(f"Gemini quota exhausted. Falling back to Ollama at {ollama_url} ...")

    if ollama_key:
        client = OpenAI(api_key=ollama_key, base_url=ollama_url)
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        resp = client.chat.completions.create(model=ollama_model or get_ollama_model(), messages=messages)
        return resp.choices[0].message.content, "ollama"

    raise ValueError("Neither GEMINI_API_KEY nor OLLAMA_API_KEY is configured.")


print("Provider ready:", "Gemini" if get_gemini_api_key() else "", "Ollama" if get_ollama_api_key() else "")

Provider ready: Gemini Ollama


## 2) Build the SQLite Database from External Files

We load the **Supplier-Parts** database from two companion files:

| File | Purpose |
|---|---|
| `SUPPLIER_PARTS_DDL.sql` | CREATE TABLE statements (schema definition) |
| `SUPPLIER_PARTS.xlsx` | Data for each table (one sheet per table) |

The database has three tables:

| Table | Purpose |
|---|---|
| `SUPPLIERS` | Supplier ID, name, status, and city |
| `PARTS` | Part ID, name, color, weight, and city |
| `SHIPMENTS` | Which supplier ships which part, with quantity |

In [3]:
DB_PATH = "supplier_parts.db"
DDL_FILE = "SUPPLIER_PARTS_DDL.sql"
XLSX_FILE = "SUPPLIER_PARTS.xlsx"

# Remove stale DB from previous runs so the demo is reproducible.
if Path(DB_PATH).exists():
    Path(DB_PATH).unlink()

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# --- 1) Execute DDL from .sql file ---
ddl_text = Path(DDL_FILE).read_text(encoding="utf-8")
# SQLite uses TEXT instead of VARCHAR2 / CHAR / NUMERIC — but it is lenient
# with type names, so the Oracle-style DDL works as-is.
cur.executescript(ddl_text)
print(f"Schema loaded from {DDL_FILE}")

# --- 2) Load data from Excel (one sheet per table) ---
wb = openpyxl.load_workbook(XLSX_FILE, read_only=True, data_only=True)

# Insertion order matters because of foreign-key constraints:
# SUPPLIERS and PARTS first, then SHIPMENTS.
table_order = ["SUPPLIERS", "PARTS", "SHIPMENTS"]

for table_name in table_order:
    ws = wb[table_name]
    rows = list(ws.iter_rows(values_only=True))
    headers = rows[0]          # first row is column names
    data_rows = rows[1:]       # remaining rows are data
    placeholders = ", ".join("?" * len(headers))
    cur.executemany(
        f"INSERT INTO {table_name} VALUES ({placeholders})",
        data_rows,
    )
    print(f"  {table_name}: {len(data_rows)} rows inserted")

wb.close()
conn.commit()
print(f"\nDatabase '{DB_PATH}' ready.")

Schema loaded from SUPPLIER_PARTS_DDL.sql
  SUPPLIERS: 6 rows inserted
  PARTS: 8 rows inserted
  SHIPMENTS: 15 rows inserted

Database 'supplier_parts.db' ready.


## 3) Schema Extraction — the "Retrieval" in DBMS RAG

In document-based RAG we retrieve text chunks.  
In DBMS RAG the **schema itself is the retrieved context** — it tells the LLM what tables, columns, and relationships exist so it can write valid SQL.

We also include a few sample rows per table so the model understands the data format.

In [4]:
def get_schema_context(db_path, sample_rows=3):
    """Extract CREATE TABLE statements and sample rows as a text block."""
    con = sqlite3.connect(db_path)
    cur = con.cursor()

    # Get all CREATE TABLE DDL
    cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = cur.fetchall()

    parts = []
    for table_name, ddl in tables:
        parts.append(f"-- Table: {table_name}")
        parts.append(ddl + ";")

        # Sample rows
        cur.execute(f"SELECT * FROM [{table_name}] LIMIT {int(sample_rows)}")
        rows = cur.fetchall()
        col_names = [desc[0] for desc in cur.description]
        parts.append(f"-- Sample rows ({table_name}): columns = {col_names}")
        for row in rows:
            parts.append(f"--   {row}")
        parts.append("")

    con.close()
    return "\n".join(parts)


schema_context = get_schema_context(DB_PATH)
print(schema_context)

-- Table: PARTS
CREATE TABLE PARTS (
    PID      CHAR(3)       ,
    Pname    VARCHAR2(50),
    Color    VARCHAR2(20),
    Weight   NUMERIC,
    Pcity    VARCHAR2(20),
    CONSTRAINT PARTS_PK PRIMARY KEY (PID) );
-- Sample rows (PARTS): columns = ['PID', 'Pname', 'Color', 'Weight', 'Pcity']
--   ('P1', 'Nut', 'Red', 12, 'London')
--   ('P2', 'Bolt', 'Green', 17, 'Paris')
--   ('P3', 'Washer', 'Blue', 17, 'Rome')

-- Table: SHIPMENTS
CREATE TABLE SHIPMENTS (
    SPID     CHAR(8)     ,
    SID      CHAR(3)     ,
    PID      CHAR(3)     ,
    Quantity INTEGER,
    CONSTRAINT SHIPMENTS_PK PRIMARY KEY (SPID, PID, SID),
    CONSTRAINT SHIPMENTS_PID_FK FOREIGN KEY (PID)
        REFERENCES PARTS,    
    CONSTRAINT SHIPMENTS_SID_FK FOREIGN KEY (SID) 
        REFERENCES SUPPLIERS (SID)  );
-- Sample rows (SHIPMENTS): columns = ['SPID', 'SID', 'PID', 'Quantity']
--   ('SP0301', 'S1', 'P1', 300)
--   ('SP0301', 'S1', 'P2', 200)
--   ('SP0301', 'S1', 'P3', 400)

-- Table: SUPPLIERS
CREATE TABLE 

## 4) Text-to-SQL Generation

This is the core RAG step: the LLM receives the schema context and the user's natural-language question, then produces a SQL query.

**Safety note:** We only allow SELECT statements. The generated SQL is validated before execution to prevent any data modification.

In [5]:
SQL_SYSTEM_PROMPT = (
    "You are an expert SQL analyst. Given the database schema below, write a "
    "single SQLite-compatible SELECT query that answers the user's question.\n"
    "Return ONLY the SQL query — no explanation, no markdown fences, no comments.\n"
    "If the question cannot be answered from the schema, reply with: SELECT 'NOT_ANSWERABLE' AS result;"
)


def extract_sql(raw_text):
    """Strip markdown fences or extra prose, returning just the SQL statement."""
    # Try to extract from ```sql ... ``` block
    match = re.search(r"```(?:sql)?\s*\n?(.*?)```", raw_text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    # Otherwise take the whole response, stripping leading/trailing whitespace
    return raw_text.strip().rstrip(";")


def generate_sql(question, schema, gemini_model="gemini-2.5-flash", ollama_model=None):
    """Ask the LLM to produce a SELECT query for the given question."""
    prompt = f"DATABASE SCHEMA:\n{schema}\n\nQUESTION:\n{question}"
    raw_response, provider = generate_text(
        prompt=prompt,
        system_prompt=SQL_SYSTEM_PROMPT,
        gemini_model=gemini_model,
        ollama_model=ollama_model,
    )
    sql = extract_sql(raw_response)
    return sql, provider


def execute_sql(db_path, sql):
    """Run a read-only SQL query and return column names + rows."""
    # Safety: only allow SELECT
    normalized = sql.strip().upper()
    if not normalized.startswith("SELECT"):
        return [], [], "Blocked: only SELECT queries are allowed."

    con = sqlite3.connect(db_path)
    try:
        cur = con.execute(sql)
        columns = [desc[0] for desc in cur.description]
        rows = cur.fetchall()
        return columns, rows, None
    except Exception as exc:
        return [], [], str(exc)
    finally:
        con.close()


print("Text-to-SQL functions ready.")

Text-to-SQL functions ready.


## 5) Answer Generation — Grounding on Query Results

After executing the SQL, we feed the **actual results** back to the LLM and ask it to compose a natural-language answer.

This is the same grounding principle as document-based RAG: the model must base its answer on retrieved evidence, not on parametric knowledge.

In [6]:
ANSWER_SYSTEM_PROMPT = (
    "You are a helpful business analyst. Use ONLY the provided SQL results to answer "
    "the user's question. If the results are empty or the query returned NOT_ANSWERABLE, "
    "say you cannot determine the answer from the available data."
)


def answer_with_dbms_rag(question, db_path=DB_PATH, gemini_model="gemini-2.5-flash", ollama_model=None):
    """Full DBMS-RAG pipeline: schema retrieval -> SQL generation -> execution -> answer."""
    schema = get_schema_context(db_path)

    print(f"  [Step 1] Schema retrieved ({len(schema)} chars)")

    sql, sql_provider = generate_sql(question, schema, gemini_model=gemini_model, ollama_model=ollama_model)
    print(f"  [Step 2] SQL generated (provider: {sql_provider})")
    print(f"           {sql}")

    columns, rows, error = execute_sql(db_path, sql)
    if error:
        print(f"  [Step 3] SQL execution error: {error}")
        return f"SQL failed: {error}", sql, sql_provider

    print(f"  [Step 3] SQL executed — {len(rows)} row(s) returned")

    # Format results for the LLM
    result_text = f"Columns: {columns}\n"
    for row in rows:
        result_text += f"  {row}\n"

    prompt = (
        f"QUESTION:\n{question}\n\n"
        f"SQL QUERY USED:\n{sql}\n\n"
        f"QUERY RESULTS:\n{result_text}"
    )
    answer, ans_provider = generate_text(
        prompt=prompt,
        system_prompt=ANSWER_SYSTEM_PROMPT,
        gemini_model=gemini_model,
        ollama_model=ollama_model,
    )
    print(f"  [Step 4] Answer generated (provider: {ans_provider})")

    return answer, sql, sql_provider


def answer_without_rag(question, gemini_model="gemini-2.5-flash", ollama_model=None):
    """Direct LLM answer with no database context."""
    answer, provider = generate_text(prompt=question, gemini_model=gemini_model, ollama_model=ollama_model)
    return answer, provider


print("RAG answer pipeline ready.")

RAG answer pipeline ready.


## 6) Run End-to-End Examples

We test several questions and compare:
- **With DBMS RAG:** schema → SQL → execute → grounded answer
- **Without RAG:** direct LLM answer (no database access)

Watch for hallucinations in the "without RAG" answers — the model has no access to actual shipment data.

In [7]:
questions = [
    "What is the total quantity of all shipments?",
    "Which supplier has shipped the most parts? Show their name and total quantity.",
    "List all parts supplied by supplier S1, with quantities.",
    "How many distinct parts are shipped from London?",
    "What is the average shipment quantity per supplier located in Paris?",
]


def preview(text, max_len=800):
    text = text or ""
    return text[:max_len] + ("..." if len(text) > max_len else "")


if not has_llm_provider():
    print("Error: Set GEMINI_API_KEY or OLLAMA_API_KEY before running.")
else:
    for i, q in enumerate(questions, start=1):
        print("\n" + "=" * 80)
        print(f"Q{i}. {q}\n")

        try:
            answer_rag, sql_used, prov = answer_with_dbms_rag(q)
            print("\n  [WITH DBMS RAG]")
            print(preview(answer_rag))
        except Exception as e:
            print(f"  RAG error: {e}")

        try:
            answer_direct, prov_direct = answer_without_rag(q)
            print(f"\n  [WITHOUT RAG] (provider: {prov_direct})")
            print(preview(answer_direct))
        except Exception as e:
            print(f"  Direct error: {e}")


Q1. What is the total quantity of all shipments?

  [Step 1] Schema retrieved (1345 chars)
  [Step 2] SQL generated (provider: gemini)
           SELECT SUM(Quantity) FROM SHIPMENTS
  [Step 3] SQL executed — 1 row(s) returned
  [Step 4] Answer generated (provider: gemini)

  [WITH DBMS RAG]
The total quantity of all shipments is 5300.

  [WITHOUT RAG] (provider: gemini)
To calculate the total quantity of all shipments, I need the individual quantities for each shipment.

Please provide the data. For example, you could give me:

*   A list of quantities (e.g., 100 units, 50 units, 230 units)
*   A table or spreadsheet with a "quantity" column.

Once you provide the information, I can sum them up for you!

Q2. Which supplier has shipped the most parts? Show their name and total quantity.

  [Step 1] Schema retrieved (1345 chars)
  [Step 2] SQL generated (provider: gemini)
           SELECT
  T1.Sname,
  SUM(T2.Quantity) AS TotalQuantity
FROM SUPPLIERS AS T1
INNER JOIN SHIPMENTS AS T2
  

### Checkpoint: Reflection Questions

1. Compare the RAG and non-RAG answers. Which ones show hallucinated numbers?
2. What happens if you ask a question about data that is **not** in the database (e.g., supplier revenue or profit)?
3. What are the risks of letting an LLM generate SQL against a production database?
4. How does this DBMS-RAG pattern compare to the vector-store RAG we used with ChromaDB?

## 7) Interactive Query (Optional)

Use the cell below to ask your own questions against the database.

In [8]:
# Change this question to anything you want to ask about the Supplier-Parts database.
my_question = "Which part color has the highest total shipment quantity?"

if has_llm_provider():
    print(f"Question: {my_question}\n")
    try:
        answer, sql, provider = answer_with_dbms_rag(my_question)
        print(f"\nSQL used:\n  {sql}")
        print(f"\nAnswer ({provider}):\n{answer}")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Set GEMINI_API_KEY or OLLAMA_API_KEY first.")

Question: Which part color has the highest total shipment quantity?

  [Step 1] Schema retrieved (1345 chars)
  [Step 2] SQL generated (provider: gemini)
           SELECT T1.Color
FROM PARTS AS T1
INNER JOIN SHIPMENTS AS T2
  ON T1.PID = T2.PID
GROUP BY
  T1.Color
ORDER BY
  SUM(T2.Quantity) DESC
LIMIT 1
  [Step 3] SQL executed — 1 row(s) returned
  [Step 4] Answer generated (provider: gemini)

SQL used:
  SELECT T1.Color
FROM PARTS AS T1
INNER JOIN SHIPMENTS AS T2
  ON T1.PID = T2.PID
GROUP BY
  T1.Color
ORDER BY
  SUM(T2.Quantity) DESC
LIMIT 1

Answer (gemini):
The part color with the highest total shipment quantity is **Green**.


## 8) Teaching Notes and Exercises

**Key takeaways:**
- RAG is not limited to vector databases — any structured data source can serve as context.
- In DBMS RAG, the **schema** is the retrieval context and **SQL** is the retrieval mechanism.
- Grounding the final answer on actual query results prevents hallucination of specific numbers.
- Loading schema from `.sql` files and data from `.xlsx` files mirrors real-world workflows.

**Exercises:**
- Add a new table (e.g., `PROJECTS` with a project-supplier-part relationship) and ask questions that require joins across it.
- Try asking ambiguous questions and observe how the LLM interprets them.
- Modify `SQL_SYSTEM_PROMPT` to request query explanations alongside the SQL.
- Compare the quality of SQL generated by different models (Gemini vs. Ollama).
- Discuss: when would you prefer DBMS RAG over vector-store RAG for a business application?